In [0]:
%run  /Shared/iplDatabricks/spn_adf_databricks_conn

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
silver_path = "abfss://silver@ipldalalakestorage.dfs.core.windows.net/"
schema_loc = "abfss://metastore@ipldalalakestorage.dfs.core.windows.net/schema_loc"
checkpoint_loc = "abfss://metastore@ipldalalakestorage.dfs.core.windows.net/checkPoint_loc"

In [0]:
# Define schema for Raw layer

schema = StructType([
    StructField("match_id",               IntegerType(), True),
    StructField("season",                 StringType(),  True),
    StructField("start_date",             DateType(),    True),
    StructField("venue",                  StringType(),  True),
    StructField("innings",                IntegerType(), True),
    StructField("ball",                   DoubleType(),  True),
    StructField("batting_team",           StringType(),  True),
    StructField("bowling_team",           StringType(),  True),
    StructField("striker",                StringType(),  True),
    StructField("non_striker",            StringType(),  True),
    StructField("bowler",                 StringType(),  True),
    StructField("runs_off_bat",           IntegerType(), True),
    StructField("extras",                 IntegerType(), True),
    StructField("wides",                  IntegerType(), True),
    StructField("noballs",                IntegerType(), True),
    StructField("byes",                   IntegerType(), True),
    StructField("legbyes",                IntegerType(), True),
    StructField("penalty",                IntegerType(), True),
    StructField("wicket_type",            StringType(),  True),
    StructField("player_dismissed",       StringType(),  True),
    StructField("other_wicket_type",      StringType(),  True),
    StructField("other_player_dismissed", StringType(),  True)
])

In [0]:
#Read incrementally using auto loader
df = (
    spark.readStream
    .format("cloudFiles")
    .option('cloudFiles.format', 'csv')
    .option('header',True)
    .schema(schema)
    .load(bronze_path)
)

In [0]:
#Replace null values with 0
null_replace_df = df.fillna(0, subset=['wides', 'noballs', 'byes', 'legbyes', 'penalty'])

#Total run for a ball by adding runs_off_bat, extras, wides, noballs, byes, legbyes, penalty
total_run_col_df = null_replace_df.withColumn('total_runs', col('runs_off_bat') + col('extras'))

In [0]:
final = (
    total_run_col_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_loc)
    .trigger(availableNow=True)
    .start(silver_path)
)
final.awaitTermination()